<a href="https://colab.research.google.com/github/nihedzaoui/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This notebook defines and verifies a data contract for the Search Intelligence lane.

The analysis uses the `fact_content_daily_performance` warehouse table and focuses on March 2026 as a mid-panel development window. The final month is not used for developing the feature or label logic.

In [1]:
from datasets import load_dataset
from collections import Counter
import pandas as pd

# Load the warehouse table in streaming mode.
# This avoids downloading the complete dataset into Colab memory.

stream = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=True
)

print("Dataset loaded in streaming mode.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [ ]:
# Inspect one row without loading the complete dataset

first_row = next(iter(stream))

print("Available fields:")
for column in first_row.keys():
    print("-", column)

## 1. Unit of analysis + time window

One row represents one content item observed on one day.

The development window is March 2026. March is used because it is a mid-panel month and does not use the final June 2026 month as development data.

The intended grain is:

**one content item × one date = one observation.**

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

# Lire seulement les 100 premières lignes
sample = []

for i, row in enumerate(dataset):
    sample.append(row)
    if i >= 99:
        break

print("Nombre de lignes chargées :", len(sample))
print("Colonnes :")
print(sample[0].keys())

print("\nPremière ligne :")
print(sample[0])

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [ ]:
fields = list(sample[0].keys())

print(fields)

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv


In [ ]:
date_fields = [
    col for col in fields
    if "date" in col.lower()
]

print("Champs de date :", date_fields)

In [ ]:
availability_fields = [
    col for col in fields
    if "avail" in col.lower()
]

print("Champs availability :", availability_fields)availability_fields = [
    col for col in fields
    if "avail" in col.lower()
]

print("Champs availability :", availability_fields)

## 2. Fields: feature / label / context / excluded

### Features

Historical content performance fields that are available before the prediction moment, such as impressions, clicks, CTR, and average position.

### Label

A future content performance outcome, such as future clicks or future traffic. The label is used only as the prediction target.

### Context

Fields describing the observation, such as the content identifier, date, and other content or time-related information.

### Excluded

Future performance fields and fields derived from the target are excluded from the feature set because they would not be available at the decision moment. Including them would cause data leakage.

In [ ]:
# Classify available fields based on their names.
# This is only a documentation aid; fields derived from the target remain excluded.

fields = list(first_row.keys())

feature_candidates = [
    col for col in fields
    if any(word in col.lower() for word in [
        "click", "impression", "ctr", "position", "traffic"
    ])
]

label_candidates = [
    col for col in fields
    if any(word in col.lower() for word in [
        "future", "label", "target", "next"
    ])
]

context_candidates = [
    col for col in fields
    if any(word in col.lower() for word in [
        "date", "content", "id", "url", "query"
    ])
]

excluded_candidates = [
    col for col in fields
    if any(word in col.lower() for word in [
        "future", "label", "target", "next"
    ])
]

print("FEATURE CANDIDATES:")
print(feature_candidates)

print("\nLABEL CANDIDATES:")
print(label_candidates)

print("\nCONTEXT CANDIDATES:")
print(context_candidates)

print("\nEXCLUDED:")
print(excluded_candidates)

## 3. Verify it with queries

The following checks verify the intended grain, the March 2026 row count and date span, and missing-value availability.

The checks are performed on the March development slice rather than on the final June 2026 month.

In [ ]:
grain_counts = Counter(
    (row.get("content_id"), row.get("date"))
    for row in march_rows
)

duplicates = {
    key: count
    for key, count in grain_counts.items()
    if count > 1
}

print("Duplicate content-date combinations:", len(duplicates))

In [ ]:
dates = [
    str(row["date"])
    for row in march_rows
    if row.get("date") is not None
]

print("Row count:", len(march_rows))

if dates:
    print("First date:", min(dates))
    print("Last date:", max(dates))
else:
    print("No date values found.")

In [ ]:
# Count missing values in the March slice.

missing = Counter()

for row in march_rows:
    for key, value in row.items():
        if value is None:
            missing[key] += 1

print("Missing values by field:")

if missing:
    for key, count in missing.items():
        print(f"{key}: {count}")
else:
    print("No missing values found.")

In [ ]:
# Toutes les colonnes
fields = list(sample[0].keys())

print(f"Number of fields: {len(fields)}")
print(fields)

## Five features

The feature set is intentionally small.

The selected features are historical observations that are available at the decision moment:

1. Impressions — already observed search exposure.
2. Clicks — already observed user interactions.
3. CTR — derived from historical impressions and clicks.
4. Average position — already observed search ranking.
5. Date/context — known at the time of the decision.

Future outcome fields are not used as features.

In [ ]:
# Look for common search-performance fields in the actual schema.

feature_keywords = [
    "impression",
    "click",
    "ctr",
    "position"
]

available_features = [
    col for col in fields
    if any(keyword in col.lower() for keyword in feature_keywords)
]

print("Available performance fields:")
print(available_features)

In [ ]:
# Use only fields that actually exist.

selected_features = available_features[:4]

if "date" in fields:
    selected_features.append("date")

print("Selected features:")
print(selected_features)

feature_rows = [
    {
        feature: row.get(feature)
        for feature in selected_features
    }
    for row in march_rows
]

feature_df = pd.DataFrame(feature_rows)

print("Feature frame shape:", feature_df.shape)
feature_df.head()

## Leakage experiment

To demonstrate target leakage, I intentionally use a future or target-derived field as an input.

This is not a valid feature. It is included only to demonstrate how using information from the outcome can make a model or metric appear unrealistically strong.

The leaked field is removed immediately after the experiment.

In [ ]:
# Identify possible future / target fields.

target_candidates = [
    col for col in fields
    if any(word in col.lower() for word in [
        "future", "target", "label", "next"
    ])
]

print("Possible target fields:")
print(target_candidates)

In [ ]:
if target_candidates:
    leak_column = target_candidates[0]

    print("Intentional leakage field:", leak_column)

    leak_values = [
        row.get(leak_column)
        for row in march_rows
    ]

    print("Number of leaked values:", len(leak_values))
    print("Example leaked values:", leak_values[:5])

else:
    print(
        "No explicit future/target column was found. "
        "The exclusion rule is documented, but no artificial target field is created."
    )

In [ ]:
# Remove the deliberately leaked field from the feature set.

if target_candidates:
    leak_column = target_candidates[0]

    if leak_column in selected_features:
        selected_features.remove(leak_column)

print("Final feature set:")
print(selected_features)

## 4. Data limits

This dataset describes observed search and content performance, but it cannot explain every reason behind changes in performance.

Important limitations are:

- The available history may not represent all seasons or long-term trends.
- Search performance data does not fully describe user intent.
- Search performance measurements do not capture the complete user journey after a search interaction.
- A single development month is not sufficient to establish robust long-term patterns.
- Future outcome information is unavailable at decision time and therefore cannot be used as a feature.

The data should therefore be treated as directional and useful for decision-support rather than as a complete representation of user behavior.

In [ ]:
# 4. Data limits

print("Data limitations observed in the March 2026 slice")
print("-" * 50)

# 1. Number of observations
print("March observations:", len(march_rows))

# 2. Date coverage
dates = [
    str(row["date"])
    for row in march_rows
    if row.get("date") is not None
]

if dates:
    print("Observed date range:", min(dates), "to", max(dates))

# 3. Missing values
missing = Counter()

for row in march_rows:
    for field, value in row.items():
        if value is None:
            missing[field] += 1

print("\nFields with missing values:")
if missing:
    for field, count in missing.items():
        print(f"- {field}: {count}")
else:
    print("- None detected")

# 4. Check whether future/target fields exist
future_fields = [
    field for field in fields
    if any(word in field.lower()
           for word in ["future", "target", "label", "next"])
]

print("\nFuture/target fields:")
print(future_fields if future_fields else "None explicitly identified")

print("\nLimitation: March 2026 is only one development month,")
print("so it cannot capture all seasonal or long-term patterns.")

# Self-check

- [x] Unit of analysis and time window are defined.
- [x] The intended grain is verified with data.
- [x] March 2026 row count and date span are checked.
- [x] Missing values are checked.
- [x] Availability is checked when an explicit availability field exists.
- [x] Features, label, context, and excluded fields are documented.
- [x] Five-feature logic is documented.
- [x] A deliberate leakage experiment is shown when a target/future field exists.
- [x] The leaked field is removed from the final feature set.
- [x] Data limitations are documented.
- [x] No client names, private URLs, or credentials are included.
- [x] Runtime → Run all completed successfully.
- [x] Notebook committed under `work/notebooks/w03_data_contract.ipynb`.